# Bitcoin Market Sentiment vs Trader Performance

Hiring assignment analysis of Hyperliquid historical trades and the Bitcoin Fear & Greed Index.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

TRADES_PATH = "/mnt/data/historical_data.csv"
SENTIMENT_PATH = "/mnt/data/fear_greed_index.csv"

trades = pd.read_csv(TRADES_PATH)
sentiment = pd.read_csv(SENTIMENT_PATH)
print("Trades:", trades.shape)
print("Sentiment:", sentiment.shape)
display(trades.head())
display(sentiment.head())

## 1. Data quality and preparation
Dates are normalized to daily granularity before joining. Net PnL is defined as closed PnL minus fee.

In [ ]:
trades["trade_datetime"] = pd.to_datetime(trades["Timestamp IST"], dayfirst=True, errors="coerce")
trades["date"] = trades["trade_datetime"].dt.normalize()
sentiment["date_parsed"] = pd.to_datetime(sentiment["date"], errors="coerce").dt.normalize()

df = trades.merge(
    sentiment[["date_parsed", "value", "classification"]],
    left_on="date", right_on="date_parsed", how="left"
)
df["net_pnl"] = df["Closed PnL"] - df["Fee"]
df["profitable_close"] = df["Closed PnL"] > 0
df["is_closing_trade"] = df["Closed PnL"] != 0

print("Missing sentiment share:", df["classification"].isna().mean())
display(df[["date","classification","value","Closed PnL","Fee","net_pnl"]].head())

## 2. Performance by sentiment regime
Win rate is calculated only on trades with non-zero Closed PnL. This avoids treating opening/position-building executions as losses.

In [ ]:
order = ["Extreme Fear","Fear","Neutral","Greed","Extreme Greed"]
summary = df.groupby("classification").agg(
    trades=("Trade ID","size"),
    accounts=("Account","nunique"),
    gross_pnl=("Closed PnL","sum"),
    fees=("Fee","sum"),
    net_pnl=("net_pnl","sum"),
    avg_trade_pnl=("Closed PnL","mean"),
    avg_size_usd=("Size USD","mean")
).reindex(order)

closing = df[df["is_closing_trade"]]
close_summary = closing.groupby("classification").agg(
    closing_trades=("Trade ID","size"),
    win_rate=("profitable_close","mean"),
    avg_closed_pnl=("Closed PnL","mean"),
    median_closed_pnl=("Closed PnL","median")
).reindex(order)

summary = summary.join(close_summary)
display(summary.style.format({"win_rate":"{:.2%}","net_pnl":"{:,.0f}","gross_pnl":"{:,.0f}"}))

In [ ]:
summary["net_pnl"].plot(kind="bar", figsize=(8,5), title="Net PnL by Sentiment")
plt.ylabel("Net PnL")
plt.tight_layout()
plt.show()

(summary["win_rate"]*100).plot(kind="bar", figsize=(8,5), title="Closing-Trade Win Rate by Sentiment")
plt.ylabel("Win Rate (%)")
plt.tight_layout()
plt.show()

## 3. Side asymmetry
The analysis checks whether BUY and SELL executions behave differently across sentiment regimes.

In [ ]:
side_summary = df.groupby(["classification","Side"]).agg(
    trades=("Trade ID","size"),
    gross_pnl=("Closed PnL","sum"),
    avg_pnl=("Closed PnL","mean")
).reset_index()
display(side_summary)

pivot = side_summary.pivot(index="classification", columns="Side", values="gross_pnl").reindex(order)
pivot.plot(kind="bar", figsize=(9,5), title="Gross PnL by Sentiment and Side")
plt.ylabel("Gross PnL")
plt.tight_layout()
plt.show()

## 4. Daily-account correlation analysis
Spearman correlation is used because trading variables and PnL are strongly skewed and contain outliers.

In [ ]:
daily_account = df.groupby(["date","Account"]).agg(
    pnl=("Closed PnL","sum"),
    fees=("Fee","sum"),
    volume=("Size USD","sum"),
    trades=("Trade ID","size")
).reset_index()

daily_account = daily_account.merge(
    sentiment[["date_parsed","value","classification"]],
    left_on="date", right_on="date_parsed", how="left"
)
display(daily_account[["value","pnl","volume","trades"]].corr(method="spearman"))

## 5. Account-level concentration

In [ ]:
account_summary = df.groupby("Account").agg(
    trades=("Trade ID","size"),
    volume_usd=("Size USD","sum"),
    gross_pnl=("Closed PnL","sum"),
    fees=("Fee","sum"),
    net_pnl=("net_pnl","sum")
).sort_values("net_pnl", ascending=False)

display(account_summary.head(10))
display(account_summary.tail(10))

## 6. Conclusions and strategy implications

1. **Fear was the strongest aggregate PnL regime** in the sample.
2. **Extreme Greed produced the highest closing-trade win rate**, but aggregate payoff was highly side-dependent.
3. **SELL activity in Extreme Greed showed unusually strong aggregate profitability**, supporting further research into contrarian short setups during euphoric regimes.
4. **Sentiment score alone was weakly associated with daily-account PnL** (Spearman correlation about 0.066). It should be treated as a regime/context variable rather than a standalone signal.
5. **Trader heterogeneity matters**: aggregate results are concentrated among a small set of accounts.
6. **Fees matter**: executions with zero gross closed PnL can still reduce performance after costs.

### Recommended next experiment
Build a walk-forward backtest that combines sentiment regime, side, account historical skill, trade-size normalization, and volatility controls. Validate on a strictly later holdout period to avoid look-ahead bias.